# Etapa 2 — Execução do Pipeline de Big Data

**Projeto:** TechPay — Monitoramento de Risco de Fraude  
**Ambiente:** rota sem privilégios administrativos  
**Tecnologias:** Python, Pandas, DuckDB, PyArrow e Parquet

## Objetivo

Executar a ingestão e a transformação da base `avaliacao_transactions.csv`
nas camadas Raw, Bronze, Silver e Gold, aplicando validações de qualidade,
particionamento físico e agregações destinadas à análise de fraude.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("DuckDB:", duckdb.__version__)
print("PyArrow:", pa.__version__)

Python: 3.13.3
Pandas: 3.0.5
DuckDB: 1.5.5
PyArrow: 25.0.1


In [2]:
# Localização das pastas do projeto
pasta_atual = Path.cwd()

if pasta_atual.name == "etapa2_bigdata":
    PASTA_ETAPA2 = pasta_atual
else:
    PASTA_ETAPA2 = (
        pasta_atual
        / "avaliacao_final"
        / "etapa2_bigdata"
    )

PASTA_DADOS = PASTA_ETAPA2 / "dados"
PASTA_RAW = PASTA_DADOS / "raw"
PASTA_BRONZE = PASTA_DADOS / "bronze"
PASTA_SILVER = PASTA_DADOS / "silver"
PASTA_GOLD = PASTA_DADOS / "gold"

ARQUIVO_RAW = PASTA_RAW / "avaliacao_transactions.csv"
BANCO_DUCKDB = PASTA_DADOS / "techpay_avaliacao.duckdb"

for pasta in [
    PASTA_RAW,
    PASTA_BRONZE,
    PASTA_SILVER,
    PASTA_GOLD,
]:
    pasta.mkdir(parents=True, exist_ok=True)

print("Pasta da Etapa 2:", PASTA_ETAPA2)
print("CSV encontrado:", ARQUIVO_RAW.exists())
print("Arquivo Raw:", ARQUIVO_RAW)

Pasta da Etapa 2: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa2_bigdata
CSV encontrado: True
Arquivo Raw: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa2_bigdata\dados\raw\avaliacao_transactions.csv


## 1. Ingestão e camada Raw

A base original foi preservada em formato CSV na camada Raw. Em seguida,
foi criada uma tabela no DuckDB com esquema explícito, garantindo que
identificadores, valores, datas e o indicador de fraude fossem interpretados
com os tipos corretos.

Nenhum registro é alterado ou descartado nesta camada. A finalidade da Raw
é manter uma representação fiel da fonte e permitir o reprocessamento do
pipeline.

In [3]:
# Conexão persistente com o banco DuckDB
con = duckdb.connect(str(BANCO_DUCKDB))

con.execute("CREATE SCHEMA IF NOT EXISTS raw")

con.execute("""
    CREATE OR REPLACE TABLE raw.transactions (
        transaction_id BIGINT,
        customer_id BIGINT,
        amount DOUBLE,
        transaction_type VARCHAR,
        channel VARCHAR,
        merchant_category VARCHAR,
        "timestamp" TIMESTAMP,
        status VARCHAR,
        risk_score DOUBLE,
        segment VARCHAR,
        credit_score INTEGER,
        is_fraud BOOLEAN
    )
""")

caminho_csv = ARQUIVO_RAW.as_posix().replace("'", "''")

con.execute(f"""
    COPY raw.transactions
    FROM '{caminho_csv}'
    (
        HEADER TRUE,
        DELIMITER ','
    )
""")

print("Tabela raw.transactions criada com sucesso.")

Tabela raw.transactions criada com sucesso.


In [4]:
validacao_raw = con.execute("""
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT transaction_id) AS ids_unicos,
        COUNT(*) - COUNT(DISTINCT transaction_id) AS ids_duplicados,
        MIN("timestamp") AS primeira_transacao,
        MAX("timestamp") AS ultima_transacao,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS total_fraudes,
        ROUND(
            100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            4
        ) AS taxa_fraude_pct
    FROM raw.transactions
""").df()

validacao_raw

,total_linhas,ids_unicos,ids_duplicados,primeira_transacao,ultima_transacao,total_fraudes,taxa_fraude_pct
0,30000,30000,0,2025-01-01,2025-12-31 23:00:00,751.0,2.5033


In [5]:
esquema_raw = con.execute("""
    DESCRIBE raw.transactions
""").df()

esquema_raw

,column_name,column_type,null,key,default,extra
0,transaction_id,BIGINT,YES,None,None,None
1,customer_id,BIGINT,YES,None,None,None
2,amount,DOUBLE,YES,None,None,None
3,transaction_type,VARCHAR,YES,None,None,None
4,channel,VARCHAR,YES,None,None,None
5,merchant_category,VARCHAR,YES,None,None,None
6,timestamp,TIMESTAMP,YES,None,None,None
7,status,VARCHAR,YES,None,None,None
8,risk_score,DOUBLE,YES,None,None,None
9,segment,VARCHAR,YES,None,None,None


## 2. Camada Bronze — limpeza e particionamento

A camada Bronze padroniza os campos textuais e aplica regras de qualidade,
sem modificar o significado original dos dados. Foram verificadas
duplicidades, campos obrigatórios, valores monetários impossíveis, limites
dos escores e valores fora dos domínios esperados.

Os registros aprovados foram armazenados em Parquet e particionados por ano
e mês da transação. Essa organização permite que consultas temporais leiam
somente os períodos necessários.

In [6]:
qualidade_raw = con.execute("""
    SELECT
        COUNT(*) AS total_linhas,

        SUM(CASE
            WHEN transaction_id IS NULL
              OR customer_id IS NULL
              OR amount IS NULL
              OR "timestamp" IS NULL
              OR channel IS NULL
              OR merchant_category IS NULL
              OR is_fraud IS NULL
            THEN 1 ELSE 0
        END) AS campos_obrigatorios_ausentes,

        SUM(CASE
            WHEN amount <= 0
            THEN 1 ELSE 0
        END) AS valores_impossiveis,

        SUM(CASE
            WHEN risk_score NOT BETWEEN 0 AND 100
            THEN 1 ELSE 0
        END) AS risk_score_invalido,

        SUM(CASE
            WHEN credit_score NOT BETWEEN 300 AND 900
            THEN 1 ELSE 0
        END) AS credit_score_invalido,

        SUM(CASE
            WHEN LOWER(TRIM(channel))
                 NOT IN ('app', 'web', 'pos', 'atm')
            THEN 1 ELSE 0
        END) AS canais_invalidos,

        SUM(CASE
            WHEN LOWER(TRIM(merchant_category))
                 NOT IN (
                     'varejo',
                     'viagem',
                     'eletronico',
                     'alimentacao',
                     'servicos',
                     'saude'
                 )
            THEN 1 ELSE 0
        END) AS categorias_invalidas,

        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS ids_duplicados

    FROM raw.transactions
""").df()

qualidade_raw

,total_linhas,campos_obrigatorios_ausentes,valores_impossiveis,risk_score_invalido,credit_score_invalido,canais_invalidos,categorias_invalidas,ids_duplicados
0,30000,0.0,0.0,0.0,0.0,0.0,0.0,0


In [7]:
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

con.execute("""
    CREATE OR REPLACE TABLE bronze.transactions AS

    WITH padronizada AS (
        SELECT
            transaction_id,
            customer_id,
            amount,
            LOWER(TRIM(transaction_type)) AS transaction_type,
            LOWER(TRIM(channel)) AS channel,
            LOWER(TRIM(merchant_category)) AS merchant_category,
            "timestamp",
            LOWER(TRIM(status)) AS status,
            risk_score,
            TRIM(segment) AS segment,
            credit_score,
            is_fraud,

            ROW_NUMBER() OVER (
                PARTITION BY transaction_id
                ORDER BY "timestamp"
            ) AS numero_registro

        FROM raw.transactions
    )

    SELECT
        transaction_id,
        customer_id,
        amount,
        transaction_type,
        channel,
        merchant_category,
        "timestamp",
        status,
        risk_score,
        segment,
        credit_score,
        is_fraud,
        YEAR("timestamp") AS transaction_year,
        MONTH("timestamp") AS transaction_month

    FROM padronizada

    WHERE numero_registro = 1
      AND transaction_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND amount IS NOT NULL
      AND "timestamp" IS NOT NULL
      AND channel IS NOT NULL
      AND merchant_category IS NOT NULL
      AND is_fraud IS NOT NULL
      AND amount > 0
      AND risk_score BETWEEN 0 AND 100
      AND credit_score BETWEEN 300 AND 900
      AND channel IN ('app', 'web', 'pos', 'atm')
      AND merchant_category IN (
          'varejo',
          'viagem',
          'eletronico',
          'alimentacao',
          'servicos',
          'saude'
      )
""")

print("Tabela bronze.transactions criada.")

Tabela bronze.transactions criada.


In [8]:
caminho_bronze = PASTA_BRONZE.as_posix().replace("'", "''")

con.execute(f"""
    COPY bronze.transactions
    TO '{caminho_bronze}'
    (
        FORMAT PARQUET,
        PARTITION_BY (
            transaction_year,
            transaction_month
        ),
        OVERWRITE_OR_IGNORE TRUE
    )
""")

print("Bronze particionada exportada para:", PASTA_BRONZE)

Bronze particionada exportada para: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa2_bigdata\dados\bronze


In [9]:
validacao_bronze = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM raw.transactions)
            AS linhas_raw,

        (SELECT COUNT(*) FROM bronze.transactions)
            AS linhas_bronze,

        (SELECT COUNT(*) FROM raw.transactions)
        -
        (SELECT COUNT(*) FROM bronze.transactions)
            AS linhas_descartadas,

        (SELECT COUNT(DISTINCT transaction_id)
         FROM bronze.transactions)
            AS ids_unicos,

        (SELECT COUNT(*)
         FROM bronze.transactions)
        -
        (SELECT COUNT(DISTINCT transaction_id)
         FROM bronze.transactions)
            AS ids_duplicados,

        (SELECT COUNT(DISTINCT
            CAST(transaction_year AS VARCHAR)
            || '-'
            || CAST(transaction_month AS VARCHAR)
         )
         FROM bronze.transactions)
            AS quantidade_particoes
""").df()

validacao_bronze

,linhas_raw,linhas_bronze,linhas_descartadas,ids_unicos,ids_duplicados,quantidade_particoes
0,30000,30000,0,30000,0,12


## 3. Camada Silver — enriquecimento

A camada Silver mantém as variáveis de canal e categoria do estabelecimento,
porque ambas podem explicar diferenças relevantes no risco de fraude.

Também foram derivadas variáveis temporais, faixas de valor e faixas de risco.
Esses atributos tornam as análises posteriores mais simples e evitam que as
mesmas regras sejam repetidas em todas as consultas da camada Gold.

In [10]:
con.execute("CREATE SCHEMA IF NOT EXISTS silver")

con.execute("""
    CREATE OR REPLACE TABLE silver.transactions AS

    SELECT
        transaction_id,
        customer_id,
        amount,
        transaction_type,
        channel,
        merchant_category,
        "timestamp",
        CAST("timestamp" AS DATE) AS transaction_date,
        transaction_year,
        transaction_month,
        DAY("timestamp") AS transaction_day,
        HOUR("timestamp") AS transaction_hour,
        DAYOFWEEK("timestamp") AS day_of_week_number,

        CASE DAYOFWEEK("timestamp")
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado'
        END AS day_of_week,

        CASE
            WHEN HOUR("timestamp") BETWEEN 0 AND 4
                THEN 'Madrugada'
            WHEN HOUR("timestamp") BETWEEN 5 AND 11
                THEN 'Manhã'
            WHEN HOUR("timestamp") BETWEEN 12 AND 17
                THEN 'Tarde'
            ELSE 'Noite'
        END AS period_of_day,

        CASE
            WHEN amount <= 50 THEN 'Até R$ 50'
            WHEN amount <= 200 THEN 'R$ 50,01 a R$ 200'
            WHEN amount <= 1000 THEN 'R$ 200,01 a R$ 1.000'
            ELSE 'Acima de R$ 1.000'
        END AS amount_range,

        CASE
            WHEN risk_score < 30 THEN 'Baixo'
            WHEN risk_score < 70 THEN 'Médio'
            ELSE 'Alto'
        END AS risk_range,

        status,
        risk_score,
        segment,
        credit_score,
        is_fraud,
        CAST(is_fraud AS INTEGER) AS fraud_indicator

    FROM bronze.transactions
""")

print("Tabela silver.transactions criada.")

Tabela silver.transactions criada.


In [11]:
caminho_silver = PASTA_SILVER.as_posix().replace("'", "''")

con.execute(f"""
    COPY silver.transactions
    TO '{caminho_silver}'
    (
        FORMAT PARQUET,
        PARTITION_BY (
            transaction_year,
            transaction_month
        ),
        OVERWRITE_OR_IGNORE TRUE
    )
""")

print("Silver particionada exportada para:", PASTA_SILVER)

Silver particionada exportada para: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa2_bigdata\dados\silver


In [12]:
validacao_silver = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM bronze.transactions)
            AS linhas_bronze,

        (SELECT COUNT(*) FROM silver.transactions)
            AS linhas_silver,

        (SELECT COUNT(*) FROM bronze.transactions)
        -
        (SELECT COUNT(*) FROM silver.transactions)
            AS diferenca,

        COUNT(DISTINCT transaction_id)
            AS ids_unicos,

        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS ids_duplicados,

        SUM(CASE
            WHEN channel IS NULL
              OR merchant_category IS NULL
              OR period_of_day IS NULL
              OR amount_range IS NULL
              OR risk_range IS NULL
            THEN 1 ELSE 0
        END) AS linhas_sem_enriquecimento,

        SUM(fraud_indicator) AS total_fraudes,

        COUNT(DISTINCT
            CAST(transaction_year AS VARCHAR)
            || '-'
            || CAST(transaction_month AS VARCHAR)
        ) AS quantidade_particoes

    FROM silver.transactions
""").df()

validacao_silver

,linhas_bronze,linhas_silver,diferenca,ids_unicos,ids_duplicados,linhas_sem_enriquecimento,total_fraudes,quantidade_particoes
0,30000,30000,0,30000,0,0.0,751.0,12


In [13]:
amostra_silver = con.execute("""
    SELECT
        transaction_id,
        amount,
        amount_range,
        channel,
        merchant_category,
        transaction_hour,
        period_of_day,
        risk_score,
        risk_range,
        is_fraud
    FROM silver.transactions
    ORDER BY "timestamp"
    LIMIT 10
""").df()

amostra_silver

,transaction_id,amount,amount_range,channel,merchant_category,transaction_hour,period_of_day,risk_score,risk_range,is_fraud
0,25607,160.41,"R$ 50,01 a R$ 200",web,eletronico,0,Madrugada,26.71,Baixo,False
1,3433,21.12,Até R$ 50,app,servicos,0,Madrugada,57.70,Médio,False
2,20910,94.62,"R$ 50,01 a R$ 200",app,viagem,0,Madrugada,42.39,Médio,False
3,16181,801.25,"R$ 200,01 a R$ 1.000",app,varejo,1,Madrugada,23.29,Baixo,False
4,15864,224.00,"R$ 200,01 a R$ 1.000",web,viagem,1,Madrugada,58.75,Médio,False
5,17379,6.91,Até R$ 50,web,eletronico,1,Madrugada,28.18,Baixo,False
6,25003,46.72,Até R$ 50,app,eletronico,2,Madrugada,24.76,Baixo,False
7,5753,179.63,"R$ 50,01 a R$ 200",pos,varejo,2,Madrugada,63.11,Médio,False
8,22089,1612.12,Acima de R$ 1.000,web,servicos,2,Madrugada,57.28,Médio,False
9,27509,91.16,"R$ 50,01 a R$ 200",app,varejo,2,Madrugada,2.03,Baixo,False


## 4. Camada Gold — indicadores e agregações

A camada Gold reúne indicadores prontos para consumo pelo dashboard e pela
área de risco. Foram criadas agregações por canal, categoria de
estabelecimento, segmento e data, além de um resumo executivo.

Essas tabelas respondem quais dimensões concentram maior risco, como as
fraudes evoluem no tempo e qual é o valor financeiro associado às
transações fraudulentas.

In [14]:
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

# Resumo executivo
con.execute("""
    CREATE OR REPLACE TABLE gold.executive_summary AS
    SELECT
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT customer_id) AS total_customers,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(AVG(amount), 2) AS average_ticket,
        SUM(fraud_indicator) AS total_frauds,
        ROUND(100.0 * AVG(fraud_indicator), 4) AS fraud_rate_pct,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS amount_at_risk
    FROM silver.transactions
""")

# Risco por canal
con.execute("""
    CREATE OR REPLACE TABLE gold.fraud_by_channel AS
    SELECT
        channel,
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT customer_id) AS total_customers,
        SUM(fraud_indicator) AS total_frauds,
        ROUND(100.0 * AVG(fraud_indicator), 4) AS fraud_rate_pct,
        ROUND(AVG(amount), 2) AS average_ticket,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS amount_at_risk
    FROM silver.transactions
    GROUP BY channel
    ORDER BY fraud_rate_pct DESC
""")

# Risco por categoria
con.execute("""
    CREATE OR REPLACE TABLE gold.fraud_by_category AS
    SELECT
        merchant_category,
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT customer_id) AS total_customers,
        SUM(fraud_indicator) AS total_frauds,
        ROUND(100.0 * AVG(fraud_indicator), 4) AS fraud_rate_pct,
        ROUND(AVG(amount), 2) AS average_ticket,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS amount_at_risk
    FROM silver.transactions
    GROUP BY merchant_category
    ORDER BY fraud_rate_pct DESC
""")

# Risco por segmento
con.execute("""
    CREATE OR REPLACE TABLE gold.fraud_by_segment AS
    SELECT
        segment,
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT customer_id) AS total_customers,
        SUM(fraud_indicator) AS total_frauds,
        ROUND(100.0 * AVG(fraud_indicator), 4) AS fraud_rate_pct,
        ROUND(AVG(amount), 2) AS average_ticket,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS amount_at_risk
    FROM silver.transactions
    GROUP BY segment
    ORDER BY fraud_rate_pct DESC
""")

# Evolução diária
con.execute("""
    CREATE OR REPLACE TABLE gold.daily_metrics AS
    SELECT
        transaction_date,
        COUNT(*) AS total_transactions,
        SUM(fraud_indicator) AS total_frauds,
        ROUND(100.0 * AVG(fraud_indicator), 4) AS fraud_rate_pct,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS amount_at_risk
    FROM silver.transactions
    GROUP BY transaction_date
    ORDER BY transaction_date
""")

print("Tabelas Gold criadas com sucesso.")

Tabelas Gold criadas com sucesso.


In [15]:
tabelas_gold = [
    "executive_summary",
    "fraud_by_channel",
    "fraud_by_category",
    "fraud_by_segment",
    "daily_metrics",
]

for tabela in tabelas_gold:
    arquivo_parquet = (
        PASTA_GOLD / f"{tabela}.parquet"
    ).as_posix().replace("'", "''")

    arquivo_csv = (
        PASTA_GOLD / f"{tabela}.csv"
    ).as_posix().replace("'", "''")

    con.execute(f"""
        COPY gold.{tabela}
        TO '{arquivo_parquet}'
        (FORMAT PARQUET)
    """)

    con.execute(f"""
        COPY gold.{tabela}
        TO '{arquivo_csv}'
        (HEADER TRUE, DELIMITER ',')
    """)

print("Tabelas Gold exportadas em Parquet e CSV.")

Tabelas Gold exportadas em Parquet e CSV.


In [16]:
display(con.execute("""
    SELECT * FROM gold.executive_summary
""").df())

display(con.execute("""
    SELECT * FROM gold.fraud_by_channel
""").df())

display(con.execute("""
    SELECT * FROM gold.fraud_by_category
""").df())

display(con.execute("""
    SELECT * FROM gold.fraud_by_segment
""").df())

,total_transactions,total_customers,total_amount,average_ticket,total_frauds,fraud_rate_pct,amount_at_risk
0,30000,7803,5207843.5,173.59,751.0,2.5033,119367.93


,channel,total_transactions,total_customers,total_frauds,fraud_rate_pct,average_ticket,amount_at_risk
0,app,12021,6225,455.0,3.7850,167.03,63909.91
1,atm,3007,2494,54.0,1.7958,178.94,12395.30
2,web,9004,5417,157.0,1.7437,174.73,31579.22
3,pos,5968,4156,85.0,1.4243,182.41,11483.50


,merchant_category,total_transactions,total_customers,total_frauds,fraud_rate_pct,average_ticket,amount_at_risk
0,viagem,3010,2485,160.0,5.3156,181.38,25966.40
1,saude,2951,2495,73.0,2.4737,179.07,11141.53
2,alimentacao,5976,4243,138.0,2.3092,176.62,21017.00
3,varejo,9047,5423,203.0,2.2438,168.51,34870.54
4,servicos,4576,3485,90.0,1.9668,181.63,14197.18
5,eletronico,4440,3421,87.0,1.9595,162.70,12175.28


,segment,total_transactions,total_customers,total_frauds,fraud_rate_pct,average_ticket,amount_at_risk
0,High-Risk,2915,2475,282.0,9.6741,177.11,42156.04
1,Standard,10645,5902,312.0,2.9310,175.67,50874.90
2,Premium,16440,6940,157.0,0.9550,171.63,26336.99


In [17]:
reconciliacao_gold = con.execute("""
    SELECT
        'Silver' AS origem,
        COUNT(*) AS total_transactions,
        SUM(fraud_indicator) AS total_frauds
    FROM silver.transactions

    UNION ALL

    SELECT
        'Gold por canal',
        SUM(total_transactions),
        SUM(total_frauds)
    FROM gold.fraud_by_channel

    UNION ALL

    SELECT
        'Gold por categoria',
        SUM(total_transactions),
        SUM(total_frauds)
    FROM gold.fraud_by_category

    UNION ALL

    SELECT
        'Gold por segmento',
        SUM(total_transactions),
        SUM(total_frauds)
    FROM gold.fraud_by_segment

    UNION ALL

    SELECT
        'Gold diária',
        SUM(total_transactions),
        SUM(total_frauds)
    FROM gold.daily_metrics
""").df()

reconciliacao_gold

,origem,total_transactions,total_frauds
0,Silver,30000.0,751.0
1,Gold por canal,30000.0,751.0
2,Gold por categoria,30000.0,751.0
3,Gold por segmento,30000.0,751.0
4,Gold diária,30000.0,751.0


In [18]:
validacao_gold_diaria = con.execute("""
    SELECT
        COUNT(*) AS quantidade_dias,
        MIN(transaction_date) AS primeira_data,
        MAX(transaction_date) AS ultima_data,
        SUM(total_transactions) AS total_transactions,
        SUM(total_frauds) AS total_frauds,
        ROUND(SUM(amount_at_risk), 2) AS amount_at_risk
    FROM gold.daily_metrics
""").df()

validacao_gold_diaria

,quantidade_dias,primeira_data,ultima_data,total_transactions,total_frauds,amount_at_risk
0,365,2025-01-01,2025-12-31,30000.0,751.0,119367.93


In [19]:
arquivos_gold = pd.DataFrame({
    "arquivo": [
        arquivo.name
        for arquivo in sorted(PASTA_GOLD.iterdir())
        if arquivo.is_file()
    ]
})

arquivos_gold

,arquivo
0,daily_metrics.csv
1,daily_metrics.parquet
2,executive_summary.csv
3,executive_summary.parquet
4,fraud_by_category.csv
5,fraud_by_category.parquet
6,fraud_by_channel.csv
7,fraud_by_channel.parquet
8,fraud_by_segment.csv
9,fraud_by_segment.parquet


In [20]:
con.close()
print("Pipeline executado e banco DuckDB encerrado com sucesso.")

Pipeline executado e banco DuckDB encerrado com sucesso.
